# SAC와 연속 제어 · 학습


## 실행 조건

<!-- exercise-guidance:reinforcement-learning/week-07/execution -->
- 필요 패키지: gymnasium·stable-baselines3·PyTorch.
- SB3 정책: `MlpPolicy`.
- Gymnasium `step`: 5개 반환값.
- 공개 환경 시각화: `gymnasium[classic-control]` 추가.

필요한 패키지가 없으면 Colab·Jupyter 코드 셀에서 `%pip install control==0.10.2 gymnasium==1.3.0 stable-baselines3==2.9.2`를 실행하세요.
<!-- /exercise-guidance:reinforcement-learning/week-07/execution -->


## 이번에 배울 내용

<!-- exercise-guidance:reinforcement-learning/week-07/learning-intro -->
연속 입력의 제어 문제에서 탐색과 입력 비용을 함께 고려하고, 단순 제어기와 비교합니다.
<!-- /exercise-guidance:reinforcement-learning/week-07/learning-intro -->


## 연속 제어

<!-- exercise-guidance:reinforcement-learning/week-07/learning-1 -->
SAC는 연속 행동의 확률적 정책을 학습하며 보상과 Entropy를 함께 고려합니다. 행동의 숫자 범위와 실제 토크·전압 등의 단위를 구분합니다.
<!-- /exercise-guidance:reinforcement-learning/week-07/learning-1 -->


## 문제의 대응

<!-- exercise-guidance:reinforcement-learning/week-07/learning-2 -->
Pendulum의 각도 표현·보상·시간 제한과 Motor의 Angular Velocity Tracking 문제를 대응시킵니다. 같은 알고리즘이라도 입력·목적이 다르면 평가 지표도 달라집니다.
<!-- /exercise-guidance:reinforcement-learning/week-07/learning-2 -->


## 풀이 예제

<!-- exercise-guidance:reinforcement-learning/week-07/learning-example -->
합성 Motor, `dt=0.1` s, 입력 $u\in[-2,2]$, 학습 Reference Angular Velocity 1입니다. 목표 0.5·1·1.5에서 평가합니다.
<!-- /exercise-guidance:reinforcement-learning/week-07/learning-example -->


In [1]:
"""Synthetic motor; not a hardware interface. dt=0.1 s, scalar speed and bounded input."""
import numpy as np
class Motor:
    def reset(self, seed=0, reference=1., noise=.01):
        self.rng = np.random.default_rng(seed)
        self.omega, self.t = 0., 0
        self.reference, self.noise = reference, noise
        return np.array([self.omega / 5, reference / 2], dtype=np.float32)
    def step(self, action):
        u = 2 * float(np.clip(np.asarray(action).item(), -1, 1))
        self.omega += .1 * (-.4 * self.omega + .8 * u + self.rng.uniform(-self.noise, self.noise))
        self.t += 1; error = self.reference - self.omega
        reward = -(error**2 + .05 * u**2)
        return np.array([self.omega / 5, self.reference / 2], dtype=np.float32), reward, abs(self.omega) > 5, self.t >= 100, {'error': error, 'u': u}

def motor_evaluate(policy, references=(.5, 1., 1.5)):
    rows=[]
    for reference in references:
        env=Motor(); obs=env.reset(100, reference)
        errors=[]; inputs=[]; total=0.
        while True:
            obs,r,terminated,truncated,info=env.step(policy(obs))
            errors.append(info['error']);inputs.append(info['u']);total+=r
            if terminated or truncated:break
        rows.append([reference, np.sqrt(np.mean(np.square(errors))),np.mean(np.square(inputs)),int(terminated),total])
    return np.array(rows)

def proportional(obs):return np.clip((2 * obs[1] - 5 * obs[0]), -1, 1)
print('reference / RMSE / mean u² / safety termination / Return')
print(motor_evaluate(proportional))

# Run the Queue or Motor definition first. Install gymnasium and stable-baselines3 if absent.
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import A2C, PPO, SAC
class QueueGym(gym.Env):
    def __init__(self):
        self.model=Queue();self.action_space=spaces.Discrete(3)
        self.observation_space=spaces.Box(0.,1.,(1,),dtype=np.float32)
    def reset(self,seed=None,options=None):
        super().reset(seed=seed)
        return np.array([self.model.reset(seed)/20],dtype=np.float32),{}
    def step(self,a):
        q,r,te,tr,info=self.model.step(a)
        return np.array([q/20],dtype=np.float32),r/20,te,tr,info
class MotorGym(gym.Env):
    def __init__(self):
        self.model=Motor();self.action_space=spaces.Box(-1.,1.,(1,),dtype=np.float32)
        self.observation_space=spaces.Box(-np.inf,np.inf,(2,),dtype=np.float32)
    def reset(self,seed=None,options=None):
        super().reset(seed=seed)
        return self.model.reset(seed),{}
    def step(self,a):return self.model.step(a)

def fit_sb3(algorithm='PPO',seed=0,steps=2048):
    cls={'A2C':A2C,'PPO':PPO,'SAC':SAC}[algorithm]
    env=MotorGym() if algorithm=='SAC' else QueueGym()
    options={'policy_kwargs':dict(net_arch=[32,32]),'seed':seed,'verbose':0,'device':'cpu'}
    if algorithm=='PPO':options.update(n_steps=128,batch_size=64,n_epochs=3)
    if algorithm=='A2C':options.update(n_steps=32)
    if algorithm=='SAC':options.update(learning_starts=100,buffer_size=5000,batch_size=64)
    model=cls('MlpPolicy',env,**options);model.learn(total_timesteps=steps)
    return model

model=fit_sb3('SAC')
print(motor_evaluate(lambda obs:model.predict(obs,deterministic=True)[0]))


reference / RMSE / mean u² / safety termination / Return
[[  0.5          0.12626501   0.07338515   0.          -1.96121101]
 [  1.           0.25292968   0.29432127   0.          -7.86894852]
 [  1.5          0.3970162    0.6158097    0.         -18.84123509]]
[[  0.5          0.81118223   0.63537708   0.         -68.97854718]
 [  1.           0.42250626   0.65318108   0.         -21.11705962]
 [  1.5          0.42756512   0.65434743   0.         -21.55293066]]


## 공식 문서·데이터

<!-- exercise-guidance:reinforcement-learning/week-07/learning-resources -->
- [Pendulum · 환경 정의](https://gymnasium.farama.org/environments/classic_control/pendulum/)
- [SB3 · SAC](https://stable-baselines3.readthedocs.io/en/master/modules/sac.html)
<!-- /exercise-guidance:reinforcement-learning/week-07/learning-resources -->


## 참고 예제

<!-- exercise-guidance:reinforcement-learning/week-07/learning-references -->
<details class="learning-references"><summary>추가 예제 보기</summary><ul><li><a data-course-reference="" href="https://chorok-daddy.github.io/niil/ko/courses/reinforcement-learning/week-10/RL_W10.preview.html">Soft Actor-Critic (SAC)</a></li></ul></details>
<!-- /exercise-guidance:reinforcement-learning/week-07/learning-references -->
